In [ ]:
# 02_sequence_dataset_builder.ipynb — Cell 0: Imports + reproducibility
import os, glob, json, random, hashlib, pickle
import numpy as np

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Cell 2: Paths + shard list
DATASET_PATH = "/content/drive/MyDrive/AIAGENT_FINAL/obtain_diamond_v3_fixdone"
OUT_DIR = "/content/drive/MyDrive/AIAGENT_FINAL/outputs_sequences"
os.makedirs(OUT_DIR, exist_ok=True)

shards = sorted(glob.glob(os.path.join(DATASET_PATH, "shard_*.npz")))
print("Total shards:", len(shards))
assert len(shards) == 30
print("First shard:", shards[0])

Total shards: 30
First shard: /content/drive/MyDrive/AIAGENT_FINAL/obtain_diamond_v3_fixdone/shard_000.npz


In [ ]:
# Cell 3: Split shards into train/val/test (reproducible)
# - Test: last 3 shards
# - Val:  next 2 shards before test
# - Train: rest
TEST_K = 3
VAL_K = 2

test_shards = shards[-TEST_K:]
val_shards  = shards[-(TEST_K + VAL_K):-TEST_K]
train_shards = shards[:-(TEST_K + VAL_K)]

print("Train shards:", len(train_shards))
print("Val shards:", len(val_shards))
print("Test shards:", len(test_shards))

assert set(train_shards).isdisjoint(set(val_shards))
assert set(train_shards).isdisjoint(set(test_shards))
assert set(val_shards).isdisjoint(set(test_shards))

Train shards: 25
Val shards: 2
Test shards: 3


In [ ]:
# Cell 4: Sequence parameters (Dreamer-style)
SEQ_LEN = 50      # training unroll length
BURN_IN = 10      # for initializing recurrent state (you'll use this in training)
STRIDE = 1        # step between consecutive sequences; keep 1 for maximum data
MIN_EP_LEN = SEQ_LEN + 1  # need at least one-step prediction inside the chunk

assert SEQ_LEN > 1
assert BURN_IN >= 0 and BURN_IN < SEQ_LEN

In [ ]:
# Cell 5: Helper — find episodes inside a shard using done flags
def episodes_from_done(done: np.ndarray):
    """
    Returns list of (start, end_inclusive) episode spans.
    done shape (N,) bool
    """
    done_idx = np.where(done)[0]
    spans = []
    start = 0
    for end in done_idx:
        spans.append((start, int(end)))
        start = int(end) + 1
    if start < len(done):
        spans.append((start, int(len(done) - 1)))
    return spans

def count_sequences_in_span(start, end, seq_len, stride):
    L = (end - start + 1)
    if L < (seq_len + 1):
        return 0
    # We need indices t..t+seq_len (inclusive) => length seq_len+1 states
    # t can go from start .. end - (seq_len)
    return 1 + ( (end - seq_len) - start ) // stride

In [ ]:
# Cell 6: Build sequence index list for a set of shards
def build_sequence_index(shard_paths, seq_len, stride=1):
    """
    Returns a list of sequence descriptors.
    Each descriptor is a dict:
      {
        "shard": "<filename.npz>",
        "t0": int,          # starting timestep in that shard
        "length": seq_len,  # unroll length (actions from t0..t0+seq_len-1)
      }
    We ensure the sequence stays inside one episode and has s[t0..t0+seq_len] available.
    """
    seqs = []
    skipped_short_eps = 0
    total_eps = 0

    for sp in shard_paths:
        d = np.load(sp)
        done = d["done"].astype(bool)
        spans = episodes_from_done(done)
        total_eps += len(spans)

        for (start, end) in spans:
            L = end - start + 1
            if L < (seq_len + 1):
                skipped_short_eps += 1
                continue

            # choose all possible starts at given stride
            last_start = end - seq_len
            for t0 in range(start, last_start + 1, stride):
                seqs.append({
                    "shard": os.path.basename(sp),
                    "t0": int(t0),
                    "length": int(seq_len),
                })

    meta = {
        "n_seqs": len(seqs),
        "n_eps": total_eps,
        "skipped_short_eps": skipped_short_eps,
    }
    return seqs, meta

In [ ]:
# Cell 7: Build train/val/test sequence indices
train_seqs, train_meta = build_sequence_index(train_shards, SEQ_LEN, stride=STRIDE)
val_seqs, val_meta     = build_sequence_index(val_shards, SEQ_LEN, stride=STRIDE)
test_seqs, test_meta   = build_sequence_index(test_shards, SEQ_LEN, stride=STRIDE)

print("TRAIN:", train_meta)
print("VAL  :", val_meta)
print("TEST :", test_meta)

TRAIN: {'n_seqs': 1609895, 'n_eps': 102, 'skipped_short_eps': 0}
VAL  : {'n_seqs': 150952, 'n_eps': 8, 'skipped_short_eps': 0}
TEST : {'n_seqs': 149650, 'n_eps': 12, 'skipped_short_eps': 0}


In [ ]:
# Cell 8: Optional downsampling for faster iteration (keep None for full scale)
# For Dreamer-style training on Colab, you may cap sequences.
CAP_TRAIN = 200_000
CAP_VAL   = None
CAP_TEST  = None

def maybe_cap(seqs, cap, seed=0):
    if cap is None or len(seqs) <= cap:
        return seqs
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(seqs), size=cap, replace=False)
    idx = np.sort(idx)
    return [seqs[i] for i in idx]

train_seqs = maybe_cap(train_seqs, CAP_TRAIN, seed=SEED+1)
val_seqs   = maybe_cap(val_seqs, CAP_VAL,   seed=SEED+2)
test_seqs  = maybe_cap(test_seqs, CAP_TEST,  seed=SEED+3)

print("After cap — n_train:", len(train_seqs), "n_val:", len(val_seqs), "n_test:", len(test_seqs))

After cap — n_train: 200000 n_val: 150952 n_test: 149650


In [ ]:
# Cell 9: Shuffle train sequences (val/test keep ordered)
rng = np.random.default_rng(SEED)
rng.shuffle(train_seqs)

In [ ]:
# Cell 10: Save outputs (pickle) + metadata (json)
def shard_list_hash(paths):
    # hash based on basenames only (stable across mount points)
    joined = "\n".join([os.path.basename(p) for p in paths]).encode("utf-8")
    return hashlib.sha256(joined).hexdigest()[:16]

meta = {
    "seed": SEED,
    "dataset_path": DATASET_PATH,
    "seq_len": SEQ_LEN,
    "burn_in": BURN_IN,
    "stride": STRIDE,
    "splits": {
        "train_shards": [os.path.basename(p) for p in train_shards],
        "val_shards": [os.path.basename(p) for p in val_shards],
        "test_shards": [os.path.basename(p) for p in test_shards],
        "train_shards_hash": shard_list_hash(train_shards),
        "val_shards_hash": shard_list_hash(val_shards),
        "test_shards_hash": shard_list_hash(test_shards),
    },
    "counts": {
        "train_sequences": len(train_seqs),
        "val_sequences": len(val_seqs),
        "test_sequences": len(test_seqs),
        "train_meta": train_meta,
        "val_meta": val_meta,
        "test_meta": test_meta,
    },
    "notes": {
        "sequence_definition": "Each item defines start t0 and length=SEQ_LEN; requires states s[t0..t0+SEQ_LEN] and actions a[t0..t0+SEQ_LEN-1].",
        "episode_boundary": "Sequences never cross done==True episode boundaries.",
    }
}

train_path = os.path.join(OUT_DIR, "train_sequences.pkl")
val_path   = os.path.join(OUT_DIR, "val_sequences.pkl")
test_path  = os.path.join(OUT_DIR, "test_sequences.pkl")
meta_path  = os.path.join(OUT_DIR, "sequences_meta.json")

with open(train_path, "wb") as f:
    pickle.dump(train_seqs, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(val_path, "wb") as f:
    pickle.dump(val_seqs, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(test_path, "wb") as f:
    pickle.dump(test_seqs, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Wrote:", train_path)
print("Wrote:", val_path)
print("Wrote:", test_path)
print("Wrote:", meta_path)

Wrote: /content/drive/MyDrive/AIAGENT_FINAL/outputs_sequences/train_sequences.pkl
Wrote: /content/drive/MyDrive/AIAGENT_FINAL/outputs_sequences/val_sequences.pkl
Wrote: /content/drive/MyDrive/AIAGENT_FINAL/outputs_sequences/test_sequences.pkl
Wrote: /content/drive/MyDrive/AIAGENT_FINAL/outputs_sequences/sequences_meta.json


In [ ]:
# Cell 11: Sanity check loader — sample a few sequences and verify shapes
# This does NOT save arrays; it just validates that indices are correct.

# map basename -> full path for fast loads
path_map = {os.path.basename(p): p for p in shards}

def load_sequence(seq_desc):
    sp = path_map[seq_desc["shard"]]
    t0 = seq_desc["t0"]
    L  = seq_desc["length"]

    d = np.load(sp)
    obs = d["obs"].astype(np.float32)
    mainhand = d["mainhand"].astype(np.float32)
    action = d["action"].astype(np.float32)
    reward = d["reward"].astype(np.float32)
    done = d["done"].astype(bool)

    state = np.concatenate([obs, mainhand], axis=1)  # (N,28)

    # sequence definition:
    # states: s[t0 .. t0+L]  => length L+1
    # actions: a[t0 .. t0+L-1] => length L
    s = state[t0:t0+L+1]
    a = action[t0:t0+L]
    r = reward[t0:t0+L]
    dn = done[t0:t0+L]  # done flags for those action steps

    return s, a, r, dn

for i in [0, 1, 2]:
    s, a, r, dn = load_sequence(train_seqs[i])
    print("seq", i, "states", s.shape, "actions", a.shape, "reward", r.shape, "done", dn.shape)
    assert s.shape == (SEQ_LEN+1, 28)
    assert a.shape == (SEQ_LEN, 15)
    assert r.shape == (SEQ_LEN,)
    # Ensure we did not cross an episode boundary:
    # done can be True only possibly at the final step if t0+L-1 includes terminal,
    # but because we built sequences within episode, dn should be all False.
    assert dn.sum() == 0, "Sequence crosses episode boundary (unexpected)."

print("Sanity checks passed.")

seq 0 states (51, 28) actions (50, 15) reward (50,) done (50,)
seq 1 states (51, 28) actions (50, 15) reward (50,) done (50,)
seq 2 states (51, 28) actions (50, 15) reward (50,) done (50,)
Sanity checks passed.


In [ ]:
# Cell 12: Estimate training batch counts (useful for planning)
# If you use DataLoader with batch_size = B sequences per batch:
import math
B = 64
n_batches = math.ceil(len(train_seqs) / B)
print("If batch_size =", B, "then train steps per 'epoch over index list' =", n_batches)

If batch_size = 64 then train steps per 'epoch over index list' = 3125


In [ ]:
# Cell 13 (optional): Write a brief text snippet for your report methods section
snippet = f"""
Sequence construction. We partition shards into train/val/test by shard ID (train={len(train_shards)}, val={len(val_shards)}, test={len(test_shards)}).
From each shard, we segment episodes using done flags and extract fixed-length sequences without crossing episode boundaries.
Each sequence has length L={SEQ_LEN} actions and L+1 states (s[t0..t0+L], a[t0..t0+L-1]).
We use burn-in of {BURN_IN} steps for recurrent state initialization during RSSM training.
"""
print(snippet.strip())

Sequence construction. We partition shards into train/val/test by shard ID (train=25, val=2, test=3).
From each shard, we segment episodes using done flags and extract fixed-length sequences without crossing episode boundaries.
Each sequence has length L=50 actions and L+1 states (s[t0..t0+L], a[t0..t0+L-1]).
We use burn-in of 10 steps for recurrent state initialization during RSSM training.


In [ ]:
# Final integrity check: all referenced shard basenames exist in DATASET_PATH
all_names = set(os.path.basename(p) for p in shards)
def check(seqs, name):
    missing = sorted({s["shard"] for s in seqs} - all_names)
    assert len(missing) == 0, f"{name}: missing shard files: {missing[:5]}"

check(train_seqs, "train")
check(val_seqs, "val")
check(test_seqs, "test")
print("All referenced shard files exist.")

All referenced shard files exist.
